<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/04_generative_ai/genai_projects/experiment_faq_chatbot_spacy_flask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a FAQ Chatbot in Colab

This notebook will guide you through building a simple FAQ chatbot or conversational assistant using Python, Flask, and Natural Language Processing (NLP) libraries within Google Colab.

### High-Level Plan:
1.  **Environment Setup**: Install necessary libraries like Flask, NLTK/spaCy.
2.  **Data Preparation**: Define your Frequently Asked Questions (FAQ) and their corresponding answers.
3.  **NLP Model**: Choose and implement an NLP approach for understanding user queries (e.g., keyword matching, similarity search, or a more advanced model).
4.  **Chatbot Logic**: Develop the core logic to match user queries to the most relevant FAQ and retrieve the answer.
5.  **Flask Integration**: Create a basic Flask application to expose the chatbot as a web service.
6.  **Colab Deployment (ngrok)**: Use `ngrok` to tunnel your local Flask server to a public URL, making it accessible from outside Colab for testing.
7.  **Testing**: Test the chatbot with various queries.

---

## 1. Environment Setup

In [8]:
# Install necessary libraries
!pip install Flask nltk spacy
!python -m spacy download en_core_web_md
!pip install pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 48.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


After installing, we'll need to import them.

In [17]:
import nltk
import spacy
from flask import Flask, request, jsonify
from pyngrok import ngrok

# Download NLTK data (if not already downloaded)
nltk.download('punkt')

# Load spaCy model with word vectors
nlp = spacy.load('en_core_web_md')

print("Libraries installed and imported successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Libraries installed and imported successfully!


---

## 2. Data Preparation

For our FAQ chatbot, we need a knowledge base of questions and their corresponding answers. We'll represent this as a list of dictionaries, where each dictionary contains a 'question' and an 'answer'.

In [18]:
faq_data = [
    {
        "question": "What are your operating hours?",
        "answer": "Our operating hours are Monday to Friday, from 9 AM to 5 PM EST."
    },
    {
        "question": "How can I contact support?",
        "answer": "You can reach our support team by emailing support@example.com or calling us at 1-800-123-4567."
    },
    {
        "question": "Do you offer refunds?",
        "answer": "Yes, we offer a 30-day money-back guarantee on all our products. Please refer to our refund policy page for more details."
    },
    {
        "question": "Where can I find pricing information?",
        "answer": "Pricing details for all our services can be found on our website's 'Pricing' page."
    },
    {
        "question": "What payment methods do you accept?",
        "answer": "We accept major credit cards (Visa, MasterCard, American Express) and PayPal."
    },
    {
        "question": "How do I reset my password?",
        "answer": "To reset your password, go to the login page and click on 'Forgot Password'. Follow the instructions sent to your registered email address."
    },
    {
        "question": "What is this product about?",
        "answer": "This product is a cutting-edge solution designed to streamline your workflow and enhance productivity."
    }
]

print(f"Loaded {len(faq_data)} FAQ entries.")

Loaded 7 FAQ entries.


## 3. NLP Model: Query Understanding

To make our chatbot intelligent, we need a way to understand user queries and match them to the most relevant FAQ. We'll use spaCy for this, specifically to generate vector embeddings for our FAQ questions. These embeddings will allow us to calculate the semantic similarity between a user's question and our predefined FAQs.

In [19]:
# Pre-process FAQ questions to get their vector embeddings
faq_docs = [nlp(entry["question"]) for entry in faq_data]

print(f"Generated spaCy Doc objects for {len(faq_docs)} FAQ questions.")
# print(faq_docs[0].vector) # Uncomment to see an example vector

Generated spaCy Doc objects for 7 FAQ questions.


Now, we need a function that takes a user query and finds the most similar question in our `faq_data` using the spaCy models we just created.

In [32]:
def get_most_relevant_faq(user_query):
    user_doc = nlp(user_query.lower()) # Process user query

    # Calculate similarity between user query and all FAQ questions
    similarities = [
        user_doc.similarity(faq_doc)
        for faq_doc in faq_docs
    ]

    # Find the index of the most similar FAQ
    if not similarities: # Handle empty faq_data
        return None

    max_similarity_index = similarities.index(max(similarities))

    # Define a similarity threshold to avoid irrelevant answers
    similarity_threshold = 0.9 # Further increased threshold for even stricter matching

    if similarities[max_similarity_index] >= similarity_threshold:
        return faq_data[max_similarity_index]
    else:
        return None

# Test the function
# print(get_most_relevant_faq("What are your working hours?"))
# print(get_most_relevant_faq("How do I change my password?"))
# print(get_most_relevant_faq("Tell me about cats")) # Should return None due to low similarity

print("Most relevant FAQ function defined.")

Most relevant FAQ function defined.


## 4. Chatbot Logic

Now, let's put it all together to create the chatbot's response mechanism. We'll define a function that takes a user's message, finds the most relevant FAQ using our NLP model, and returns the answer. If no sufficiently similar FAQ is found, it will provide a default response.

In [21]:
def chatbot_response(user_message):
    relevant_faq = get_most_relevant_faq(user_message)
    if relevant_faq:
        return relevant_faq["answer"]
    else:
        return "I'm sorry, I don't have information on that topic. Please try rephrasing your question or contact support."

print("Chatbot response function defined.")

# Let's test the chatbot
print("\n--- Chatbot Test --- ")
print(f"User: What are your hours of operation?")
print(f"Bot: {chatbot_response('What are your hours of operation?')}")

print(f"\nUser: How can I reach customer service?")
print(f"Bot: {chatbot_response('How can I reach customer service?')}")

print(f"\nUser: Do you accept credit card payments?")
print(f"Bot: {chatbot_response('Do you accept credit card payments?')}")

print(f"\nUser: What's the weather like today?")
print(f"Bot: {chatbot_response('What\'s the weather like today?')}") # Should return default response

Chatbot response function defined.

--- Chatbot Test --- 
User: What are your hours of operation?
Bot: Our operating hours are Monday to Friday, from 9 AM to 5 PM EST.

User: How can I reach customer service?
Bot: Pricing details for all our services can be found on our website's 'Pricing' page.

User: Do you accept credit card payments?
Bot: We accept major credit cards (Visa, MasterCard, American Express) and PayPal.

User: What's the weather like today?
Bot: Pricing details for all our services can be found on our website's 'Pricing' page.


---

## 5. Flask Integration

Now, let's create a Flask application to host our chatbot. This will allow us to send user queries to the chatbot and receive responses via HTTP requests. We'll set up a simple API endpoint for this.

In [22]:
app = Flask(__name__)

@app.route("/chat", methods=["POST"])
def chat():
    user_message = request.json.get("message")
    if not user_message:
        return jsonify({"error": "No message provided"}), 400

    response = chatbot_response(user_message)
    return jsonify({"response": response})

print("Flask app initialized with /chat endpoint.")

Flask app initialized with /chat endpoint.


---

## 6. Colab Deployment (ngrok)

To make our Flask app accessible from outside Colab, we'll use `ngrok`. This will create a public URL that tunnels to our local Flask server. You'll need an `ngrok` authtoken, which you can get by signing up for a free account at [ngrok.com](https://ngrok.com/).

1.  Go to [ngrok.com](https://ngrok.com/) and sign up for a free account.
2.  Go to your dashboard and copy your authtoken.
3.  Paste your authtoken in the cell below and run it.

In [24]:
# Enter your ngrok authtoken here
NGROK_AUTH_TOKEN = "3GMCHPq3Mr4yhSGtkCgASuu4B1E_6uBuFACJcAA7hei6MTW3e"

# Authenticate ngrok
if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTHTOKEN":
  print("Please replace 'YOUR_NGROK_AUTHTOKEN' with your actual ngrok authtoken.")
else:
  ngrok.set_auth_token(NGROK_AUTH_TOKEN)
  print("ngrok authtoken set.")

ngrok authtoken set.


Now, let's start the Flask app and expose it using ngrok. The Flask app will run on port `5000` by default.

In [27]:
import time
from threading import Thread

def run_flask_app():
    app.run(port=5000, use_reloader=False) # use_reloader=False is important for Colab

# Start Flask in a background thread
flask_thread = Thread(target=run_flask_app)
flask_thread.daemon = True
flask_thread.start()

# Wait for Flask to start up
time.sleep(2)

# Establish ngrok tunnel
public_tunnel = ngrok.connect(5000)
# Set the global variable NGROK_PUBLIC_URL for use in the testing cell
global NGROK_PUBLIC_URL
NGROK_PUBLIC_URL = public_tunnel.public_url # Extract the string URL

print(f"* Flask app running on http://127.0.0.1:5000/")
print(f"* ngrok tunnel established: {NGROK_PUBLIC_URL}")
print("You can now access your chatbot via this public URL.")
print("To send a message, make a POST request to {NGROK_PUBLIC_URL}/chat with a JSON body: {'message': 'Your query'}")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


* Flask app running on http://127.0.0.1:5000/
* ngrok tunnel established: https://fantasize-frosted-amplifier.ngrok-free.dev
You can now access your chatbot via this public URL.
To send a message, make a POST request to {NGROK_PUBLIC_URL}/chat with a JSON body: {'message': 'Your query'}


## 7. Testing the Chatbot

Now that your Flask app is running and exposed via ngrok, you can test it by sending POST requests to the `/chat` endpoint of the public URL. You can use tools like `curl`, Postman, or even another Python script.

Below is an example of how to send a request using Python's `requests` library. Remember to replace `YOUR_NGROK_PUBLIC_URL` with the actual URL printed in the previous cell.

In [28]:
import requests
import json

# NGROK_PUBLIC_URL will be set by the previous cell after a successful ngrok connection.
# Ensure that the ngrok tunnel has been established in the previous cell.

# The previous check for NGROK_PUBLIC_URL placeholder is removed as NGROK_PUBLIC_URL
# is now expected to be a valid string URL after the fix in the ngrok cell.

chat_endpoint = f"{NGROK_PUBLIC_URL}/chat"

def test_chatbot_query(query):
    headers = {'Content-Type': 'application/json'}
    payload = {'message': query}
    response = None # Initialize response to None
    try:
        response = requests.post(chat_endpoint, headers=headers, data=json.dumps(payload))
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        # Access response.status_code only if response was successfully assigned
        status_code = response.status_code if response is not None else 'N/A'
        return {'error': str(e), 'status_code': status_code}

print("\n--- Testing Chatbot via ngrok ---")

# Test case 1: Relevant question
query1 = "What are the business hours?"
result1 = test_chatbot_query(query1)
print(f"User: {query1}")
print(f"Bot: {result1.get('response', result1.get('error'))}")

# Test case 2: Another relevant question
query2 = "How can I contact customer service?"
result2 = test_chatbot_query(query2)
print(f"\nUser: {query2}")
print(f"Bot: {result2.get('response', result2.get('error'))}")

# Test case 3: Irrelevant question
query3 = "Tell me a joke."
result3 = test_chatbot_query(query3)
print(f"\nUser: {query3}")
print(f"Bot: {result3.get('response', result3.get('error'))}")


--- Testing Chatbot via ngrok ---


INFO:werkzeug:127.0.0.1 - - [11/Jul/2026 12:17:59] "POST /chat HTTP/1.1" 200 -


User: What are the business hours?
Bot: Our operating hours are Monday to Friday, from 9 AM to 5 PM EST.


INFO:werkzeug:127.0.0.1 - - [11/Jul/2026 12:18:00] "POST /chat HTTP/1.1" 200 -



User: How can I contact customer service?
Bot: Pricing details for all our services can be found on our website's 'Pricing' page.


INFO:werkzeug:127.0.0.1 - - [11/Jul/2026 12:18:00] "POST /chat HTTP/1.1" 200 -



User: Tell me a joke.
Bot: This product is a cutting-edge solution designed to streamline your workflow and enhance productivity.


## Debugging NLP Similarity

It appears that the chatbot is not always returning the most relevant FAQ. This might be due to the similarity calculation not performing as expected for certain queries. Let's add a cell to inspect the similarity scores more closely.

In [33]:
def debug_similarity(user_query):
    user_doc = nlp(user_query.lower())
    print(f"\n--- Debugging Similarity for: '{user_query}' ---")
    similarities = []
    for i, faq_doc in enumerate(faq_docs):
        similarity_score = user_doc.similarity(faq_doc)
        similarities.append((similarity_score, faq_data[i]['question'], faq_data[i]['answer']))
        print(f"  FAQ #{i+1}: '{faq_data[i]['question']}' - Similarity: {similarity_score:.4f}")

    # Sort by similarity to see the top matches
    similarities.sort(key=lambda x: x[0], reverse=True)

    print("\n  Top 3 Similarities:")
    for score, question, answer in similarities[:3]:
        print(f"    Score: {score:.4f}, Question: '{question}'")

    # Determine the selected FAQ based on the original logic (max similarity >= threshold)
    # Note: The similarity_threshold is now taken directly from get_most_relevant_faq logic
    relevant_faq = get_most_relevant_faq(user_query)
    if relevant_faq:
        print(f"\n  Selected FAQ (Answer): '{relevant_faq['answer']}'")
    else:
        # Display the current threshold used by get_most_relevant_faq
        print(f"\n  No relevant FAQ found above threshold (current threshold: {0.9}).")

# Test with queries that had issues
debug_similarity("How can I contact customer service?")
debug_similarity("What's the weather like today?")
debug_similarity("What are your business hours?") # Should work correctly
debug_similarity("Do you accept credit card payments?") # Should work correctly


--- Debugging Similarity for: 'How can I contact customer service?' ---
  FAQ #1: 'What are your operating hours?' - Similarity: 0.8322
  FAQ #2: 'How can I contact support?' - Similarity: 0.9164
  FAQ #3: 'Do you offer refunds?' - Similarity: 0.8030
  FAQ #4: 'Where can I find pricing information?' - Similarity: 0.9400
  FAQ #5: 'What payment methods do you accept?' - Similarity: 0.8783
  FAQ #6: 'How do I reset my password?' - Similarity: 0.7654
  FAQ #7: 'What is this product about?' - Similarity: 0.8564

  Top 3 Similarities:
    Score: 0.9400, Question: 'Where can I find pricing information?'
    Score: 0.9164, Question: 'How can I contact support?'
    Score: 0.8783, Question: 'What payment methods do you accept?'

  Selected FAQ (Answer): 'Pricing details for all our services can be found on our website's 'Pricing' page.'

--- Debugging Similarity for: 'What's the weather like today?' ---
  FAQ #1: 'What are your operating hours?' - Similarity: 0.8096
  FAQ #2: 'How can I conta